# Autofocus

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

In [ ]:
import numpy as np
import cv2

from matplotlib.figure import Figure
from matplotlib.patches import Circle

import panel as pn

from enderscope.serial import list_ports, Stage
from enderleaf.preview_panel import preview
from enderleaf.image import to_pil, crop_image, Rectangle, lap_var

In [ ]:
%matplotlib widget

In [ ]:
pn.extension("ace", "jsoneditor")

In [ ]:
preview().start_video()

In [ ]:
# preview().camera.set_controls({"AwbEnable": True, "AeEnable": False})
# preview().camera.set_controls({"ExposureTime": 50000, "AnalogueGain": 3.0})

In [ ]:
# list available serial ports
ports = list_ports()
for port in ports:
    if "USB" in port.description:
        stage_port = port
        print("* " + str(port))
    else:
        print("  " + str(port))

In [ ]:
s = Stage(stage_port.device, 115200)

In [ ]:
s.home(debug=True)

In [ ]:
preview().set_crop(top=600, bottom=550, left=1500, right=1670)

In [ ]:
s.move_position([134.25, 125.0, 36.0], debug=True)

In [ ]:
zrange = np.array(range(-5, 5, 1)) * 1
zrange

In [ ]:
preview().camera.set_controls(
    {"LensPosition": preview().camera.camera_controls["LensPosition"][1]}
)

In [ ]:
preview().show()

In [ ]:
pos = s.get_position()
mxScore = -1
bestZ = 0
variances = {}
images = {}

for z in zrange:
    s.move_position([pos[0], pos[1], z + pos[2]])
    s.finish_moves()
    img = preview().do_capture_array(
        crop_data=Rectangle(top=600, bottom=2592 - 550, left=1500, right=4608 - 1670)
    )
    images[z + pos[2]] = img
    grayImage = (
        np.float32(img[:, :, 0]) + np.float32(img[:, :, 1]) + np.float32(img[:, :, 2])
    ) / 3
    score = lap_var(grayImage)
    variances[z + pos[2]] = score
    if score > mxScore:
        mxScore = score
        bestZ = z

s.move_position([pos[0], pos[1], pos[2]])

In [ ]:
sel_image = pn.widgets.DiscreteSlider(
    name="Select image",
    options=list(images.keys()),
    value=pos[2] + bestZ,
    sizing_mode="stretch_width",
)
ph_image = pn.pane.Placeholder()
ph_plot = pn.pane.Placeholder()


@pn.depends(sel_image.param.value, watch=True)
def on_index_changed(index):
    ph_image.object = to_pil(images[index]).resize((600, 600))

    fig = Figure(figsize=(4, 4))
    ax = fig.subplots(nrows=1, ncols=1)
    fig.suptitle("Variance")
    ax.plot(list(variances.keys()), list(variances.values()))
    ax.add_patch(
        Circle(
            xy=(pos[2] + bestZ, variances[pos[2] + bestZ]),
            radius=1,
            edgecolor="lime",
            facecolor="lime",
            linewidth=1,
        )
    )
    ax.add_patch(
        Circle(
            xy=(index, variances[index]),
            radius=0.7,
            edgecolor="lightblue",
            facecolor="blue",
            linewidth=1,
        )
    )
    ax.set_xlabel("Z")
    ax.set_ylabel("Variance")
    ph_plot.object = fig


on_index_changed(sel_image.value)

pn.Column(pn.Row(ph_image, ph_plot), sel_image)